### Workshop the steps in a synthetic reconstruction of narG variants. 

First, build two faa's, one to be used as a reference, the other to generate raw reads.

In [ ]:
ref_directory = '../data/whole_genomes/K00370_rep.faa'

from Bio import SeqIO
import random

# Read all sequences
records = list(SeqIO.parse(ref_directory, "fasta"))
print(f"Total sequences: {len(records)}")

# Shuffle and split 70/30
random.shuffle(records)
split_point = int(0.89 * len(records)) #0.89 makes it so there are exactly 100 spike-in sequences

reference_set = records[:split_point]
spikein_set = records[split_point:]

# Write outputs
SeqIO.write(reference_set, "../out/reconstruction/reference.faa", "fasta")
SeqIO.write(spikein_set, "../out/reconstruction/spikein.faa", "fasta")

print(f"Reference: {len(reference_set)} sequences")
print(f"Spike-in: {len(spikein_set)} sequences")

Total sequences: 906
Reference: 806 sequences
Spike-in: 100 sequences


Next, generate reads. 

In [6]:
from Bio import SeqIO
import random

def generate_paired_end_reads(sequence, read_length=150, coverage=30, insert_size=300):
    """Generate paired-end reads with proper insert size"""
    reads_r1 = []
    reads_r2 = []
    seq_length = len(sequence)
    
    # Calculate number of read pairs needed
    num_pairs = (seq_length * coverage) // (2 * read_length)
    
    for _ in range(num_pairs):
        # Random start position for R1
        start_r1 = random.randint(0, seq_length - insert_size)
        end_r1 = start_r1 + read_length
        start_r2 = start_r1 + insert_size - read_length
        
        # Ensure we don't go beyond sequence boundaries
        if start_r2 + read_length <= seq_length:
            read_r1 = sequence[start_r1:end_r1]
            read_r2 = sequence[start_r2:start_r2 + read_length]
            
            # R2 is reverse complement (simulating actual sequencing)
            read_r2 = reverse_complement(read_r2)
            
            reads_r1.append(read_r1)
            reads_r2.append(read_r2)
    
    return reads_r1, reads_r2

def reverse_complement(seq):
    """Simple reverse complement"""
    comp = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G', 'N': 'N'}
    return ''.join(comp.get(base, base) for base in reversed(seq))

# Read spike-in sequences
spikein_records = list(SeqIO.parse("../out/reconstruction/spikein.faa", "fasta"))

# Generate paired-end reads
all_r1 = []
all_r2 = []

for record in spikein_records:
    r1, r2 = generate_paired_end_reads(str(record.seq), read_length=150, coverage=15)  # 15x each = 30x total
    all_r1.extend(r1)
    all_r2.extend(r2)

print(f"Generated {len(all_r1)} read pairs")

# Write R1 and R2 files
with open("../out/reconstruction/spikein_R1.fastq", "w") as f1, \
     open("../out/reconstruction/spikein_R2.fastq", "w") as f2:
    
    for i, (read1, read2) in enumerate(zip(all_r1, all_r2)):
        # R1 file
        f1.write(f"@read_{i}/1\n")
        f1.write(f"{read1}\n")
        f1.write("+\n")
        f1.write("I" * len(read1) + "\n")
        
        # R2 file  
        f2.write(f"@read_{i}/2\n")
        f2.write(f"{read2}\n")
        f2.write("+\n")
        f2.write("I" * len(read2) + "\n")

print("Paired-end reads written to:")
print("  ../out/reconstruction/spikein_R1.fastq")
print("  ../out/reconstruction/spikein_R2.fastq")

Generated 5972 read pairs
Paired-end reads written to:
  ../out/reconstruction/spikein_R1.fastq
  ../out/reconstruction/spikein_R2.fastq
